# Embodied Motion Flow - Kaggle Bridge
Notebook compatto: training + showcase 15s con CFG/EMA.


In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = os.environ.get("EMF_REPO_URL", "https://github.com/your-org/Embodied-Motion-Flow.git")
WORKDIR = Path("/kaggle/working/Embodied-Motion-Diffusion")
PYTHON = sys.executable
def sh(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
print("Python:", PYTHON)
print("Workdir:", WORKDIR)
print("Repo URL:", REPO_URL)
print("Ready for setup")


In [ ]:
if WORKDIR.exists():
    sh(["rm", "-rf", str(WORKDIR)])
sh(["git", "clone", REPO_URL, str(WORKDIR)])
os.chdir(WORKDIR)
sh([PYTHON, "-m", "pip", "install", "-q", "-U", "pip"])
sh([PYTHON, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("Repo ready:", WORKDIR)
print("Current dir:", Path.cwd())
print("Requirements installed")
print("Step 2 completed")
print("Next: data/env mapping")
print("-")


In [ ]:
import glob, json
candidates = glob.glob("/kaggle/**/motions", recursive=True)
aist_motions = sorted([p for p in candidates if "aist" in p.lower()])
AISTPP_ROOT = aist_motions[0] if aist_motions else "/kaggle/working/data/aist-smpl-clean/motions"
AIST_BASE = str(Path(AISTPP_ROOT).parent)
os.environ["AISTPP_ROOT"] = AISTPP_ROOT
os.environ["AISTPP_SPLIT_ROOT"] = f"{AIST_BASE}/splits"
os.environ["AISTPP_IGNORE_LIST"] = f"{AIST_BASE}/ignore_list.txt"
os.environ["SHOWCASE_TRACK"] = os.environ.get("SHOWCASE_TRACK", "/kaggle/working/data/stardust.wav")
print("AIST root:", os.environ["AISTPP_ROOT"])
print("Track:", os.environ["SHOWCASE_TRACK"])
print(json.dumps({k: os.environ[k] for k in ["AISTPP_SPLIT_ROOT","AISTPP_IGNORE_LIST"]}, indent=2))


In [ ]:
import yaml
cfg = yaml.safe_load(Path("config.yaml").read_text())
cfg["project"]["output_dir"] = "/kaggle/working/outputs"
cfg["device"]["preference"] = "auto"
cfg["training"]["auto_resume"] = True
cfg["training"]["mixed_precision"] = True
cfg["inference"]["generation_frames"] = 450
cfg["showcase"]["clip_start_seconds"] = 46.0
cfg["showcase"]["clip_duration_seconds"] = 15.0
Path("config.kaggle.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Wrote config.kaggle.yaml")
print("Config patch completed")


In [ ]:
RUN_TRAIN = True
CHECKPOINT = "/kaggle/working/outputs/checkpoints/model.pt"
cmd = [PYTHON, "kaggle_showcase_main.py", "--config", "config.kaggle.yaml"]
if not RUN_TRAIN:
    cmd += ["--skip-train", "--checkpoint", CHECKPOINT]
print("RUN_TRAIN:", RUN_TRAIN)
print("Checkpoint:", CHECKPOINT)
print("Command:", " ".join(cmd))
sh(cmd)
print("Pipeline completed")
print("Artifacts in /kaggle/working/outputs")
print("Step 5 completed")


In [ ]:
from IPython.display import Video, display
out = Path("/kaggle/working/outputs/showcase")
for p in sorted(out.glob("*")):
    print(p)
viral = out / "stardust_0046_0101_viral.mp4"
research = out / "stardust_0046_0101_research.mp4"
print("Viral exists:", viral.exists())
print("Research exists:", research.exists())
if viral.exists():
    display(Video(str(viral), embed=True, width=480))
if research.exists():
    display(Video(str(research), embed=True, width=720))
